# Pilot fine-tuning

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime ayarlarından T4 GPU açılmalı.")

CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes scikit-learn pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.0 MB/s eta 0:00:00


In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print("Libraries imported.")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Libraries imported.
CUDA available: True
GPU: Tesla T4


In [ ]:
train_path = "/data/processed/strategy_b/train.csv"
validation_path = "/data/processed/strategy_b/validation.csv"
test_path = "/data/processed/strategy_b/test.csv"

print("Train exists:", os.path.exists(train_path))
print("Validation exists:", os.path.exists(validation_path))
print("Test exists:", os.path.exists(test_path))

Train exists: True
Validation exists: True
Test exists: True


In [ ]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(validation_df["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())

Train shape: (11575, 8)
Validation shape: (1447, 8)
Test shape: (1447, 8)

Columns:
['input_text', 'label', 'Score', 'soru', 'context', 'cevap', 'kaynak', 'veri türü']

Train label distribution:
label
0    7085
1    4490
Name: count, dtype: int64

Validation label distribution:
label
0    886
1    561
Name: count, dtype: int64

Test label distribution:
label
0    885
1    562
Name: count, dtype: int64


In [ ]:
train_dataset = Dataset.from_pandas(train_df[["input_text", "label"]])
validation_dataset = Dataset.from_pandas(validation_df[["input_text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["input_text", "label"]])

print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['input_text', 'label'],
    num_rows: 11575
})
Dataset({
    features: ['input_text', 'label'],
    num_rows: 1447
})
Dataset({
    features: ['input_text', 'label'],
    num_rows: 1447
})


In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", model_id)
print("Pad token:", tokenizer.pad_token)
print("Pad token id:", tokenizer.pad_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: Qwen/Qwen2.5-1.5B-Instruct
Pad token: <|endoftext|>
Pad token id: 151643


In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", model_id)
print("Pad token:", tokenizer.pad_token)
print("Pad token id:", tokenizer.pad_token_id)

Tokenizer loaded: Qwen/Qwen2.5-1.5B-Instruct
Pad token: <|endoftext|>
Pad token id: 151643


In [ ]:
MAX_LENGTH = 1024

def tokenize_function(example):
    return tokenizer(
        example["input_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_validation = validation_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_validation = tokenized_validation.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

tokenized_train.set_format("torch")
tokenized_validation.set_format("torch")
tokenized_test.set_format("torch")

print(tokenized_train)
print(tokenized_validation)
print(tokenized_test)

Map:   0%|          | 0/11575 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 11575
})
Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 1447
})
Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 1447
})


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded for sequence classification:", model_id)
print("Number of labels:", model.config.num_labels)
print("Pad token id:", model.config.pad_token_id)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded for sequence classification: Qwen/Qwen2.5-1.5B-Instruct
Number of labels: 2
Pad token id: 151643


In [ ]:
for name, module in model.named_modules():
    if "proj" in name or "score" in name:
        print(name)

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.2.mlp.gate_proj
model.layers.2.mlp.up_proj
model.layers.2.mlp.down_proj
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.3.mlp.gate_proj
model.layers.3.mlp.up_proj
model.layers.3.mlp.down_proj
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.4.mlp.g

In [ ]:
!pip install -U torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.7 MB/s eta 0:00:00


In [ ]:
import torchao
print(torchao.__version__)

0.17.0


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none"
)

model = get_peft_model(model, lora_config)

print("LoRA applied.")
model.print_trainable_parameters()

LoRA applied.
trainable params: 9,235,456 || all params: 1,552,952,832 || trainable%: 0.5947


In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator ready.")

Data collator ready.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)
    weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

print("Metrics function ready.")

Metrics function ready.


In [ ]:
output_dir = "/content/qwen_strategy_b_lora"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=False,
    bf16=False,
    report_to="none",

    save_total_limit=2
)

print("Training arguments ready.")

Training arguments ready.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready.")

Trainer ready.


In [ ]:
small_eval_dataset = tokenized_validation.select(range(50))

small_eval_results = trainer.evaluate(small_eval_dataset)

print("Small validation sanity check:")
print(small_eval_results)

Small validation sanity check:
{'eval_loss': 2.1861374378204346, 'eval_model_preparation_time': 0.0345, 'eval_accuracy': 0.38, 'eval_macro_f1': 0.2989597467209407, 'eval_weighted_f1': 0.24175486205336952, 'eval_runtime': 15.8825, 'eval_samples_per_second': 3.148, 'eval_steps_per_second': 3.148}


In [ ]:
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")

GPU memory cleared.


In [ ]:
MAX_LENGTH = 768

def tokenize_function(example):
    return tokenizer(
        example["input_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_validation = validation_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_validation = tokenized_validation.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

tokenized_train.set_format("torch")
tokenized_validation.set_format("torch")
tokenized_test.set_format("torch")

print(tokenized_train)
print(tokenized_validation)
print(tokenized_test)

Map:   0%|          | 0/11575 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 11575
})
Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 1447
})
Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 1447
})


In [ ]:
small_validation_dataset = tokenized_validation.select(range(300))

print(small_validation_dataset)

Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 300
})


In [ ]:
output_dir = "/content/qwen_strategy_b_lora_pilot_100"

training_args = TrainingArguments(
    output_dir=output_dir,

    max_steps=100,
    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,

    logging_strategy="steps",
    logging_steps=10,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=False,
    bf16=False,
    report_to="none",

    save_total_limit=2
)

print("Pilot 100-step training arguments ready.")

Pilot 100-step training arguments ready.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=small_validation_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Pilot trainer ready.")

Pilot trainer ready.


In [ ]:
train_result = trainer.train()

print("Pilot training completed.")

Step,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
50,16.548338,1.133588,0.576667,0.524932,0.561512
100,18.345087,1.110050,0.563333,0.527138,0.557664


Pilot training completed.


In [ ]:
test_results = trainer.evaluate(tokenized_test)

print("Test results:")
print(test_results)

Test results:
{'eval_loss': 1.127575397491455, 'eval_accuracy': 0.5473393227366966, 'eval_macro_f1': 0.5192103318150528, 'eval_weighted_f1': 0.5451693720084556, 'eval_runtime': 349.8472, 'eval_samples_per_second': 4.136, 'eval_steps_per_second': 4.136, 'epoch': 0.13822894168466524}


In [ ]:
test_predictions_output = trainer.predict(tokenized_test)

test_logits = test_predictions_output.predictions
test_predictions = np.argmax(test_logits, axis=-1)

test_prediction_df = test_df.copy()
test_prediction_df["prediction"] = test_predictions
test_prediction_df["correct"] = test_prediction_df["label"] == test_prediction_df["prediction"]

print(test_prediction_df[["label", "prediction", "correct", "Score", "soru"]].head())

print("\nCorrect count:")
print(test_prediction_df["correct"].value_counts())

print("\nPrediction distribution:")
print(test_prediction_df["prediction"].value_counts().sort_index())

   label  prediction  correct  Score  \
0      0           0     True      8   
1      1           0    False      9   
2      1           0    False      9   
3      0           0     True      8   
4      1           1     True      9   

                                                soru  
0  Türklerin dünya tarihine olan etkisi hangi ala...  
1  Eğitim bakanlığının teşkilat ve görevleri hakk...  
2  Bir öğrenci, derslerin ötesinde kendini nasıl ...  
3                        Sınav sonrası ne yapılmalı?  
4  Bütüncül bir eğitim anlayışı nasıl tanımlanabi...  

Correct count:
correct
True     792
False    655
Name: count, dtype: int64

Prediction distribution:
prediction
0    912
1    535
Name: count, dtype: int64


In [ ]:
os.makedirs("/outputs/tables", exist_ok=True)

qwen_finetune_pilot_results = pd.DataFrame([
    {
        "model": "Qwen2.5-1.5B-Instruct",
        "training_type": "lora_finetuning_pilot",
        "dataset_strategy": "strategy_b",
        "max_length": MAX_LENGTH,
        "max_steps": training_args.max_steps,
        "learning_rate": training_args.learning_rate,
        "train_batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
        "test_accuracy": test_results.get("eval_accuracy"),
        "test_macro_f1": test_results.get("eval_macro_f1"),
        "test_weighted_f1": test_results.get("eval_weighted_f1"),
        "prediction_label_0_count": int((test_prediction_df["prediction"] == 0).sum()),
        "prediction_label_1_count": int((test_prediction_df["prediction"] == 1).sum()),
        "correct_count": int(test_prediction_df["correct"].sum()),
        "wrong_count": int((~test_prediction_df["correct"]).sum())
    }
])

qwen_finetune_pilot_results.to_csv(
    "/outputs/tables/qwen_finetune_pilot_results.csv",
    index=False
)

test_prediction_df.to_csv(
    "/outputs/tables/qwen_finetune_pilot_test_predictions.csv",
    index=False
)

print("Saved:")
print("/outputs/tables/qwen_finetune_pilot_results.csv")
print("/outputs/tables/qwen_finetune_pilot_test_predictions.csv")

qwen_finetune_pilot_results

Saved:
/outputs/tables/qwen_finetune_pilot_results.csv
/outputs/tables/qwen_finetune_pilot_test_predictions.csv


,model,training_type,dataset_strategy,max_length,max_steps,learning_rate,train_batch_size,gradient_accumulation_steps,test_accuracy,test_macro_f1,test_weighted_f1,prediction_label_0_count,prediction_label_1_count,correct_count,wrong_count
0,Qwen2.5-1.5B-Instruct,lora_finetuning_pilot,strategy_b,768,100,0.00002,1,16,0.547339,0.51921,0.545169,912,535,792,655


In [ ]:
print(os.path.exists("/outputs/tables/qwen_finetune_pilot_results.csv"))
print(os.path.exists("/outputs/tables/qwen_finetune_pilot_test_predictions.csv"))

True
True


# Qwen QLoRA Long-run Fine-Tuning with Google Drive Checkpoints

Önemli:
- Bu bölüm kendi içinde yeniden ortamı hazırlar.
- Runtime restart olduktan sonra da yukarıdaki pilot hücrelerine bağımlı kalmadan çalıştırılabilir.
- Checkpointler `/content` yerine Google Drive'a kaydedilir.
- GPU yoksa model yükleme ve eğitim hücrelerini çalıştırma.


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

if not torch.cuda.is_available():
    print("GPU yok. Bu bölümde eğitim başlatma.")


CUDA available: True
GPU: Tesla T4


## 1. Paket kurulumu


In [2]:
!pip install transformers datasets accelerate peft bitsandbytes scikit-learn pandas -q
!pip install -U torchao -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.2 MB/s eta 0:00:00


## 2. Importlar


In [3]:
import os
import gc
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

print("Imports ready.")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


Imports ready.
CUDA available: True
GPU: Tesla T4


## 3. Google Drive bağlantısı ve klasörler


In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
drive_project_dir = "/content/drive/MyDrive/turkish_answer_quality_slm"
drive_tables_dir = "/content/drive/MyDrive/turkish_answer_quality_slm/outputs/tables"
drive_model_dir = "/content/drive/MyDrive/turkish_answer_quality_slm/qwen_strategy_b_qlora_longrun"

os.makedirs(drive_project_dir, exist_ok=True)
os.makedirs(drive_tables_dir, exist_ok=True)
os.makedirs(drive_model_dir, exist_ok=True)

print("Drive project dir:", drive_project_dir)
print("Tables dir:", drive_tables_dir)
print("Model checkpoint dir:", drive_model_dir)


Drive project dir: /content/drive/MyDrive/turkish_answer_quality_slm
Tables dir: /content/drive/MyDrive/turkish_answer_quality_slm/outputs/tables
Model checkpoint dir: /content/drive/MyDrive/turkish_answer_quality_slm/qwen_strategy_b_qlora_longrun


## 4. Veri dosyalarını yükleme

Colab Files panelinde şu klasör yapısı olmalı:

`/data/processed/strategy_b/`

İçinde şu üç dosya olmalı:

- train.csv
- validation.csv
- test.csv


In [7]:
train_path = "/data/processed/strategy_b/train.csv"
validation_path = "/data/processed/strategy_b/validation.csv"
test_path = "/data/processed/strategy_b/test.csv"

print("Train exists:", os.path.exists(train_path))
print("Validation exists:", os.path.exists(validation_path))
print("Test exists:", os.path.exists(test_path))


Train exists: True
Validation exists: True
Test exists: True


In [8]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(validation_df["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())


Train shape: (11575, 8)
Validation shape: (1447, 8)
Test shape: (1447, 8)

Train label distribution:
label
0    7085
1    4490
Name: count, dtype: int64

Validation label distribution:
label
0    886
1    561
Name: count, dtype: int64

Test label distribution:
label
0    885
1    562
Name: count, dtype: int64


## 5. Dataset ve tokenizer


In [9]:
train_dataset = Dataset.from_pandas(train_df[["input_text", "label"]])
validation_dataset = Dataset.from_pandas(validation_df[["input_text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["input_text", "label"]])

print(train_dataset)
print(validation_dataset)
print(test_dataset)


Dataset({
    features: ['input_text', 'label'],
    num_rows: 11575
})
Dataset({
    features: ['input_text', 'label'],
    num_rows: 1447
})
Dataset({
    features: ['input_text', 'label'],
    num_rows: 1447
})


In [10]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", model_id)
print("Pad token:", tokenizer.pad_token)
print("Pad token id:", tokenizer.pad_token_id)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: Qwen/Qwen2.5-1.5B-Instruct
Pad token: <|endoftext|>
Pad token id: 151643


## 6. Tokenization


In [11]:
MAX_LENGTH = 768

def tokenize_function(example):
    return tokenizer(
        example["input_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_validation = validation_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_validation = tokenized_validation.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

tokenized_train.set_format("torch")
tokenized_validation.set_format("torch")
tokenized_test.set_format("torch")

print(tokenized_train)
print(tokenized_validation)
print(tokenized_test)


Map:   0%|          | 0/11575 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 11575
})
Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 1447
})
Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 1447
})


## 7. Qwen modelini 4-bit QLoRA için yükleme

Bu bölüm normal LoRA yerine 4-bit quantized model yükler. Amaç RAM/GPU bellek kullanımını azaltmaktır.


In [12]:
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id

print("4-bit Qwen model loaded for sequence classification.")


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


4-bit Qwen model loaded for sequence classification.


## 8. Modeli k-bit training için hazırlama ve LoRA uygulama


In [13]:
model = prepare_model_for_kbit_training(model)
print("Model prepared for k-bit training.")


Model prepared for k-bit training.


In [14]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none"
)

model = get_peft_model(model, lora_config)

print("QLoRA applied.")
model.print_trainable_parameters()


QLoRA applied.
trainable params: 9,235,456 || all params: 1,552,952,832 || trainable%: 0.5947


## 9. Data collator ve metrikler


In [15]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)
    weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

print("Data collator and metrics ready.")


Data collator and metrics ready.


## 10. Long-run training ayarları


In [16]:
small_validation_dataset = tokenized_validation.select(range(300))

output_dir = drive_model_dir

training_args = TrainingArguments(
    output_dir=output_dir,

    max_steps=600,
    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200,

    logging_strategy="steps",
    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=False,
    bf16=False,
    report_to="none",

    save_total_limit=3
)

print("Long-run QLoRA training arguments ready.")
print("Output dir:", output_dir)
print(small_validation_dataset)


Long-run QLoRA training arguments ready.
Output dir: /content/drive/MyDrive/turkish_answer_quality_slm/qwen_strategy_b_qlora_longrun
Dataset({
    features: ['input_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 300
})


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=small_validation_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Long-run QLoRA trainer ready.")


Long-run QLoRA trainer ready.


## 11. Eğitimi başlatma


In [18]:
train_result = trainer.train()

print("Long-run QLoRA training completed.")


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
200,13.930333,0.914903,0.546667,0.515578,0.544212
400,13.201636,0.764481,0.530000,0.488976,0.522760
600,12.021022,0.719606,0.603333,0.545518,0.583341


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Long-run QLoRA training completed.


In [19]:
import os

checkpoint_dir = "/content/drive/MyDrive/turkish_answer_quality_slm/qwen_strategy_b_qlora_longrun"

print(os.listdir(checkpoint_dir))

['checkpoint-200', 'checkpoint-400', 'checkpoint-600']


## 12. Test değerlendirme


In [20]:
test_results_longrun = trainer.evaluate(tokenized_test)

print("Long-run QLoRA test results:")
print(test_results_longrun)


Long-run QLoRA test results:
{'eval_loss': 0.7547430396080017, 'eval_accuracy': 0.5715272978576365, 'eval_macro_f1': 0.5071417584107939, 'eval_weighted_f1': 0.5469056766542549, 'eval_runtime': 516.8357, 'eval_samples_per_second': 2.8, 'eval_steps_per_second': 2.8, 'epoch': 0.8293736501079914}


## 13. Test prediction detayları


In [21]:
test_predictions_output = trainer.predict(tokenized_test)

test_logits = test_predictions_output.predictions
test_predictions = np.argmax(test_logits, axis=-1)

test_prediction_df = test_df.copy()
test_prediction_df["prediction"] = test_predictions
test_prediction_df["correct"] = test_prediction_df["label"] == test_prediction_df["prediction"]

print(test_prediction_df[["label", "prediction", "correct", "Score", "soru"]].head())

print("\nCorrect count:")
print(test_prediction_df["correct"].value_counts())

print("\nPrediction distribution:")
print(test_prediction_df["prediction"].value_counts().sort_index())

print("\nTrue label distribution:")
print(test_prediction_df["label"].value_counts().sort_index())

   label  prediction  correct  Score  \
0      0           1    False      8   
1      1           0    False      9   
2      1           0    False      9   
3      0           1    False      8   
4      1           0    False      9   

                                                soru  
0  Türklerin dünya tarihine olan etkisi hangi ala...  
1  Eğitim bakanlığının teşkilat ve görevleri hakk...  
2  Bir öğrenci, derslerin ötesinde kendini nasıl ...  
3                        Sınav sonrası ne yapılmalı?  
4  Bütüncül bir eğitim anlayışı nasıl tanımlanabi...  

Correct count:
correct
True     827
False    620
Name: count, dtype: int64

Prediction distribution:
prediction
0    1085
1     362
Name: count, dtype: int64

True label distribution:
label
0    885
1    562
Name: count, dtype: int64


## 14. Sonuçları Drive'a kaydetme


In [22]:
import os
import pandas as pd

drive_tables_dir = "/content/drive/MyDrive/turkish_answer_quality_slm/outputs/tables"
os.makedirs(drive_tables_dir, exist_ok=True)

qwen_qlora_longrun_results = pd.DataFrame([
    {
        "model": "Qwen2.5-1.5B-Instruct",
        "training_type": "qlora_finetuning_longrun",
        "dataset_strategy": "strategy_b",
        "max_length": MAX_LENGTH,
        "max_steps": training_args.max_steps,
        "learning_rate": training_args.learning_rate,
        "train_batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
        "test_accuracy": test_results_longrun.get("eval_accuracy"),
        "test_macro_f1": test_results_longrun.get("eval_macro_f1"),
        "test_weighted_f1": test_results_longrun.get("eval_weighted_f1"),
        "prediction_label_0_count": int((test_prediction_df["prediction"] == 0).sum()),
        "prediction_label_1_count": int((test_prediction_df["prediction"] == 1).sum()),
        "true_label_0_count": int((test_prediction_df["label"] == 0).sum()),
        "true_label_1_count": int((test_prediction_df["label"] == 1).sum()),
        "correct_count": int(test_prediction_df["correct"].sum()),
        "wrong_count": int((~test_prediction_df["correct"]).sum()),
        "checkpoint_dir": "/content/drive/MyDrive/turkish_answer_quality_slm/qwen_strategy_b_qlora_longrun"
    }
])

qwen_results_path = os.path.join(drive_tables_dir, "qwen_qlora_longrun_results.csv")
qwen_predictions_path = os.path.join(drive_tables_dir, "qwen_qlora_longrun_test_predictions.csv")

qwen_qlora_longrun_results.to_csv(qwen_results_path, index=False)
test_prediction_df.to_csv(qwen_predictions_path, index=False)

print("Saved:")
print(qwen_results_path)
print(qwen_predictions_path)

qwen_qlora_longrun_results

Saved:
/content/drive/MyDrive/turkish_answer_quality_slm/outputs/tables/qwen_qlora_longrun_results.csv
/content/drive/MyDrive/turkish_answer_quality_slm/outputs/tables/qwen_qlora_longrun_test_predictions.csv


,model,training_type,dataset_strategy,max_length,max_steps,learning_rate,train_batch_size,gradient_accumulation_steps,test_accuracy,test_macro_f1,test_weighted_f1,prediction_label_0_count,prediction_label_1_count,true_label_0_count,true_label_1_count,correct_count,wrong_count,checkpoint_dir
0,Qwen2.5-1.5B-Instruct,qlora_finetuning_longrun,strategy_b,768,600,0.00002,1,16,0.571527,0.507142,0.546906,1085,362,885,562,827,620,/content/drive/MyDrive/turkish_answer_quality_...


In [23]:
print(os.path.exists(qwen_results_path))
print(os.path.exists(qwen_predictions_path))


True
True


## 15. Not

Eğitim sırasında runtime koparsa checkpointler şu Drive klasöründe kalır:

`/content/drive/MyDrive/turkish_answer_quality_slm/qwen_strategy_b_qlora_longrun`

Bir sonraki oturumda checkpointten devam etmek gerekirse bu klasör kullanılacaktır.
